In [1]:
import json
import os
import pandas as pd

def load_json_data(file_path):
    """Reads and returns data from a JSON file."""
    if not os.path.exists(file_path):
        print(f"Error: File '{file_path}' not found in the current directory.")
        return None

    with open(file_path, 'r') as file:
        return json.load(file)

def inspect_csv_data(file_path):
    df = pd.read_csv(file_path)
    num_data, num_features = df.shape
    return num_data, num_features

def compute_num_params_nn(all_layers, layer_in, layer_out):
    total_params = 0
    total_layers = [layer_in] + all_layers + [layer_out]
    for i in range(len(total_layers) - 1):
        n_in = total_layers[i]
        n_out = total_layers[i+1]

        # Calculation formula: (Inputs * Outputs) + Outputs
        weights = n_in * n_out
        biases = n_out
        subtotal = weights + biases
        total_params += subtotal
    return total_params

In [2]:
"""Extracts and prints key metrics from the specified JSON files."""
# Define file names
name_list = [
        'AgNP',
        'CO2RRLCA',
        'CO2RRNPV',
        'CO2RRMSP',
        'CO2HPx10',
        'CO2HEx10',
]

data_performance = {'name': [], 'model': [], 'R2': [], 'num_params': [], 'num_features': [], 'num_datapoints': []}

data_params_mlp = []
data_params_kan = []
dir_project = r"D:\pykan\github\workflows\Hyein"

for name in name_list:
    mlp_file = os.path.join("material_nn_models", name, f"{name}_metrics.json")
    kan_file = os.path.join("material_kan_models", name, f"{name}_kan_metrics.json")
    data_path = os.path.join(dir_project, 'data', f"{name}.csv")

    # Load data
    mlp_data = load_json_data(mlp_file)
    kan_data = load_json_data(kan_file)

    n_data, n_cols = inspect_csv_data(data_path)
    n_in = n_cols - 1
    n_out = 1

    # Process Standard Neural Network Data
    if mlp_data:
        mlp_params = mlp_data.get('best_params')
        nn_layers = mlp_params.get("hidden_layer_sizes")
        nn_params = compute_num_params_nn(nn_layers, n_in, n_out)

        data_performance['name'].append(name)
        data_performance['num_features'].append(n_cols)
        data_performance['num_datapoints'].append(n_data)
        data_performance['model'].append('NN')
        data_performance['R2'].append(mlp_data.get('test_r2', 0))
        data_performance['num_params'].append(nn_params)

        data_params_mlp.append({
            'name': name,
            'R2': mlp_data.get('test_r2', 0),
            'num_params': nn_params,
            'hidden_layer_sizes': nn_layers,
            'alpha': mlp_params.get('alpha', 0),
            'learning_rate_init': mlp_params.get('learning_rate_init', 0),
        })

    # Process KAN / Symbolic Regression Data
    if kan_data:
        kan_params = kan_data.get('best_params')
        kan_num_params = n_in * n_out * (kan_params['grid'] + kan_params['k'] + 1)

        data_performance['name'].append(name)
        data_performance['num_features'].append(n_cols)
        data_performance['num_datapoints'].append(n_data)
        data_performance['model'].append('KAN')
        data_performance['R2'].append(kan_data.get('test_r2', 0))
        data_performance['num_params'].append(kan_num_params)

        data_params_kan.append({
            'name': name,
            'R2': kan_data.get('test_r2', 0),
            'num_params': kan_num_params,
            'grid': kan_params.get('grid', 0),
            'steps': kan_params.get('steps', 0),
            'lamb': kan_params.get('lamb', 0),
            'lamb_coef': kan_params.get('lamb_coef', 0),
            'lamb_coefdiff': kan_params.get('lamb_coefdiff', 0),
            'lamb_entropy': kan_params.get('lamb_entropy', 0),
            'lr': kan_params.get('lr', 0),
            'sym_range': kan_params.get('sym_range', 0),
        })

df_performance = pd.DataFrame(data_performance)
df_performance.to_csv(os.path.join(dir_project, 'figures_for_paper', 'performance.csv'))
print(df_performance)

df_params_mlp = pd.DataFrame(data_params_mlp)
df_params_mlp.to_csv(os.path.join(dir_project, 'figures_for_paper', 'params_mlp.csv'))
df_params_kan = pd.DataFrame(data_params_kan)
df_params_kan.to_csv(os.path.join(dir_project, 'figures_for_paper', 'params_kan.csv'))
print(df_params_mlp)
print(df_params_kan)

        name model        R2  num_params  num_features  num_datapoints
0       AgNP    NN  0.941127        3009             6            3295
1       AgNP   KAN  0.812329          70             6            3295
2   CO2RRLCA    NN  0.982547        1281             9            3677
3   CO2RRLCA   KAN  0.965041          72             9            3677
4   CO2RRNPV    NN  0.994874        4801             9            3677
5   CO2RRNPV   KAN  0.971202         112             9            3677
6   CO2RRMSP    NN  0.989171        9473             9            3677
7   CO2RRMSP   KAN  0.915158         112             9            3677
8   CO2HPx10    NN  0.834993        3713            17            6949
9   CO2HPx10   KAN -0.295055         144            17            6949
10  CO2HEx10    NN  0.974814        2305            17            6949
11  CO2HEx10   KAN  0.973103         144            17            6949
       name        R2  num_params hidden_layer_sizes  alpha   
0      AgNP  0

In [14]:
name = 'CO2HPx10'
seed_list = [i for i in range(10)]

data_performance = {'name': [], 'seed': [], 'model': [], 'R2': [], 'num_params': []}
for seed in seed_list:
    dir_project = r"D:\pykan\github\workflows\Hyein"
    mlp_file = os.path.join("material_nn_models", name+f"_seed_{seed}", f"{name}_metrics.json")
    kan_file = os.path.join("material_kan_models", name+f"_seed_{seed}", f"{name}_kan_metrics.json")
    data_path = os.path.join(dir_project, 'data', f"{name}.csv")

    # Load data
    mlp_data = load_json_data(mlp_file)
    kan_data = load_json_data(kan_file)

    n_cols = inspect_csv_data(data_path)
    n_in = n_cols - 1
    n_out = 1

    # Process Standard Neural Network Data
    if mlp_data:
        nn_layers = mlp_data.get('best_params').get("hidden_layer_sizes")
        print(nn_layers)
        nn_params = compute_num_params_nn(nn_layers, n_in, n_out)

        data_performance['name'].append(name)
        data_performance['seed'].append(seed)
        data_performance['model'].append('NN')
        data_performance['R2'].append(mlp_data.get('test_r2', 0))
        data_performance['num_params'].append(nn_params)

    # Process KAN / Symbolic Regression Data
    if kan_data:
        kan_params = kan_data.get('best_params')
        kan_params = n_in * n_out * (kan_params['grid'] + kan_params['k'] + 1)

        data_performance['name'].append(name)
        data_performance['seed'].append(seed)
        data_performance['model'].append('KAN')
        data_performance['R2'].append(kan_data.get('test_r2', 0))
        data_performance['num_params'].append(kan_params)
df_performance = pd.DataFrame(data_performance)
print(df_performance)

       name  seed model        R2  num_params
0  CO2HPx10     0    NN  0.834723        3713
1  CO2HPx10     1    NN  0.835082        3713
2  CO2HPx10     2    NN  0.844408        3985
3  CO2HPx10     3    NN  0.840174        3985
4  CO2HPx10     4    NN  0.839968        3985
5  CO2HPx10     5    NN  0.840036        3985
6  CO2HPx10     6    NN  0.827737       10497
7  CO2HPx10     7    NN  0.840024        3985
8  CO2HPx10     8    NN  0.828249       10497
9  CO2HPx10     9    NN  0.840271        3985
